# CV Matcher - LoRA Fine-Tuning
## Qwen2.5-1.5B-Instruct ile CV Parsing + CV-JD Matching

---
## Hücre 1: Kütüphaneleri Yükle
▶️ butonuna tıkla, bitene kadar bekle (~2 dk)

In [ ]:
!pip install -q transformers accelerate peft datasets bitsandbytes scipy sentencepiece
print("Kutuphaneler hazir")

---
## Hücre 2: Projeyi Çek
▶️ tıkla (~1 dk)

In [ ]:
!rm -rf cvmatcher
!git clone https://github.com/ruveydagundogan/cvmatcher.git
%cd cvmatcher
print("Proje hazir")

---
## Hücre 3: CV Parse Modelini Eğit
▶️ tıkla, **~10-15 dk sürer**, bekle

In [ ]:
BASE = "Qwen/Qwen2.5-1.5B-Instruct"

!python backend/finetune/train_lora.py \
    --base-model $BASE \
    --data backend/finetune/data/cv_parse_dataset.json \
    --output-dir /content/cvmatcher-lora/cv-parser-v1 \
    --mode cv-parse \
    --epochs 5 \
    --batch-size 4 \
    --max-length 512 \
    --lr 2e-4 \
    --quantize

print("\nCV Parse modeli fine-tune edildi!")

---
## Hücre 4: CV-JD Match Modelini Eğit
▶️ tıkla, **~10-15 dk sürer**, bekle

In [ ]:
!python backend/finetune/train_lora.py \
    --base-model $BASE \
    --data backend/finetune/data/cv_jd_match_dataset.json \
    --output-dir /content/cvmatcher-lora/cv-jd-matcher-v1 \
    --mode cv-jd-match \
    --epochs 5 \
    --batch-size 4 \
    --max-length 512 \
    --lr 2e-4 \
    --quantize

print("\nCV-JD Match modeli fine-tune edildi!")

---
## Hücre 5: CV Coach Modelini Eğit
▶️ tıkla, **~10-15 dk sürer**, bekle. Chat asistanının Türkçe/İngilizce kaliteli cevap verebilmesi için bu adım şart.

In [ ]:
!python backend/finetune/train_lora.py \
    --base-model $BASE \
    --data backend/finetune/data/cv_coach_dataset.json \
    --output-dir /content/cvmatcher-lora/cv-coach-v1 \
    --mode cv-coach \
    --epochs 5 \
    --batch-size 4 \
    --max-length 512 \
    --lr 2e-4 \
    --quantize

print("\nCV Coach modeli fine-tune edildi!")

---
## Hücre 6: Base Model vs Fine-Tuned Karşılaştırması
▶️ tıkla (~1 dk)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

test_cv = """Python backend developer with 4 years experience.
Skilled in Django, FastAPI, PostgreSQL, Redis, Celery, Docker.
Built REST APIs serving 50K requests per minute.
Bachelor's in Software Engineering."""

messages = [
    {"role": "user", "content": f"Parse the following CV text and extract structured information: skills, experience, education, and a brief summary.\n\n{test_cv}"}
]

print("=" * 60)
print("BASE MODEL TESTI")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(BASE)
base_model = AutoModelForCausalLM.from_pretrained(BASE, device_map="auto", torch_dtype=torch.bfloat16)

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = base_model.generate(**inputs, max_new_tokens=256, temperature=0.1)
base_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(base_response[-500:] if len(base_response) > 500 else base_response)

In [ ]:
print("=" * 60)
print("FINE-TUNED MODEL (LoRA) TESTI")
print("=" * 60)

finetuned = PeftModel.from_pretrained(base_model, "/content/cvmatcher-lora/cv-parser-v1")
finetuned.eval()

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = finetuned.generate(**inputs, max_new_tokens=256, temperature=0.1)
ft_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(ft_response[-500:] if len(ft_response) > 500 else ft_response)

---
## Hücre 7: Adapter'ları İndir
▶️ tıkla, zip otomatik bilgisayarına iner

In [ ]:
import zipfile, os

zip_path = "/content/cvmatcher-lora.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk("/content/cvmatcher-lora"):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, "/content/cvmatcher-lora")
            zf.write(file_path, arcname)

print(f"Zip: {zip_path}")
from google.colab import files
files.download(zip_path)

---
## Mac'te Ollama'ya Yükleme

İndirdiğin zip'i terminalde kur:
```bash
bash backend/finetune/setup_ollama.sh ~/Downloads/cvmatcher-lora.zip
```

Not: Gemma HF onayı gelince `BASE = "Qwen/Qwen2.5-1.5B-Instruct"` satırını `BASE = "google/gemma-2b-it"` yapıp aynı notebook'u tekrar çalıştırabilirsin.